In [181]:
import numpy as np
import scipy
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

In [182]:
property = pd.read_csv("/content/sample_data/Data/properties_2022.csv")
sales = pd.read_csv("/content/sample_data/Data/sales.csv")
var_dict = pd.read_csv("/content/sample_data/Data/variable_dictionary.csv")

/tmp/ipykernel_8342/73101949.py:1: DtypeWarning: Columns (37) have mixed types. Specify dtype option on import or set low_memory=False.
  property = pd.read_csv("/content/sample_data/Data/properties_2022.csv")


Check the circumstance of missing value in property and
sales csv. Check whether the missing value fraction in
property csv approximately equals to that in dictionary


In [183]:
print (property.info())
print ("\nThe shape of property dataset is: {}\n".format(property.shape))

act_missing = property.drop(columns='property_id').isnull().mean()

var_dict["actual_missing_fraction"] = var_dict["variable"].map(act_missing)
difference = (var_dict["actual_missing_fraction"] - var_dict["missing_fraction"])
great_diff = difference[difference>1e-6]

unmatched = act_missing.index[~act_missing.index.isin(var_dict["variable"])]

if great_diff.empty and unmatched.empty:
  print ("All null fraction in property csv can match with that in dictionary\n and features are all included in dictionary")
else:
  print (great_diff)
  print (unmatched)

value_null = property.isnull().sum().sort_values(ascending=False)
print ("\nSum of missing value:\n", value_null[value_null!=0])


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 98899 entries, 0 to 98898
Data columns (total 98 columns):
 #   Column                                                 Non-Null Count  Dtype  
---  ------                                                 --------------  -----  
 0   property_id                                            98899 non-null  int64  
 1   loc_property_city                                      98897 non-null  object 
 2   loc_property_zip                                       98517 non-null  float64
 3   loc_longitude                                          98899 non-null  float64
 4   loc_latitude                                           98899 non-null  float64
 5   char_yrblt                                             98898 non-null  float64
 6   char_bldg_sf                                           98899 non-null  float64
 7   char_land_sf                                           98899 non-null  float64
 8   char_beds                                     

In [184]:
print (sales.info())
print ("\nThe shape of property dataset is: {}\n".format(sales.shape))
value_null2 = sales.isnull().sum().sort_values(ascending=False)
print (value_null2[value_null2!=0])


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 109307 entries, 0 to 109306
Data columns (total 4 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   property_id     109307 non-null  int64  
 1   transaction_id  109307 non-null  int64  
 2   sale_date       109307 non-null  object 
 3   sale_price      109307 non-null  float64
dtypes: float64(1), int64(2), object(1)
memory usage: 3.3+ MB
None

The shape of property dataset is: (109307, 4)

Series([], dtype: int64)


According to the above, there are a large amount of missing
value in property csv, and their fraction can match to
that in dictionary. No missing value exist in sales csv.


---


Confirm whether there are records out of required time

In [185]:
sales['sale_date'] = pd.to_datetime(sales['sale_date'])

# Confirm whether there are records out of required time
outoftime = sales[(sales['sale_date'] >= '2026-01-01')|(sales['sale_date'] <= '2022-12-31')]

if outoftime.empty:
  print ('All records in sale csv fall within 2023 and 2025\n')
else:
  print ('There are sale records out of specified time:\n{}'.format(outoftime))

print("char_yrblt range:", property["char_yrblt"].min(),property["char_yrblt"].max())
print("Later than 2022 char_built:",(property["char_yrblt"]>2022).sum())
print("Null or negative record:",(property["char_yrblt"] <= 0).sum())
print("Too early (<1800) :", (property["char_yrblt"]< 1800).sum())

print("\nRange of year Median Occupied Household:",property["acs5_median_household_total_occupied_year_built"].min(),
      property["acs5_median_household_total_occupied_year_built"].max())
print("Later than 2022 Median Occupied Household:",
 (property["acs5_median_household_total_occupied_year_built"] > 2022).sum())


All records in sale csv fall within 2023 and 2025

char_yrblt range: 1847.0 2021.0
Later than 2022 char_built: 0
Null or negative record: 0
Too early (<1800) : 0

Range of year Median Occupied Household: 1940.0 2007.0
Later than 2022 Median Occupied Household: 0


All records in sales csv and property csv are correctly
recorded.


---


We then test whether duplicate records and repeated
sales of the same property exist or not.

:

In [186]:
duplicate_trans_id = sales['transaction_id'].value_counts()
duplicate_list = duplicate_trans_id[duplicate_trans_id>1]
if duplicate_list.empty:
  print ('No duplicated transaction id exist.')
else:
  print ('There are duplicated transaction id existing:\n{}'.format(duplicate_list))

print ("Number of unique transaction id in sales csv: ",sales['transaction_id'].nunique())
print ("Number of unique property id in sales csv: ",sales['property_id'].nunique())
print ("Number of unique property id in property csv: ",property['property_id'].nunique())

print ("Sales shape: ",sales.shape)
print ("Property shape: ",property.shape)

# 将property id出现的统计值添加回原始DataFrame，进行分组
sales["sale_count"] = sales.groupby("property_id")["transaction_id"].transform("count")
# boolean shows property id是否重复销售
sales["is_repeat_sale"] = sales["sale_count"] > 1

print (sales["sale_count"].min()== 1)
print ((sales["is_repeat_sale"] == (sales["sale_count"] > 1)).all())

# # 检查是否存在重复的property id中，有同一日期的不同销售记录
# repeat_sales = sales[sales["is_repeat_sale"] == True]
# result = (repeat_sales.groupby("property_id")["sale_date"].nunique().le(1).all())
# print(result)

No duplicated transaction id exist.
Number of unique transaction id in sales csv:  109307
Number of unique property id in sales csv:  98899
Number of unique property id in property csv:  98899
Sales shape:  (109307, 4)
Property shape:  (98899, 98)
True
True


From the above, we can find that in the property csv,
all of the property id is unique. However, in the sales
csv, the number of property id does not equal to the
row number, which means that there are duplicated property id
existing. Transaction id is unique. This means that one
property id may match mulitple transaction id, which is
compliant.



---


Set the first column 'property_id' as index

In [187]:
property = property.set_index ('property_id')
print (property.head())
sales = sales.set_index ('property_id')
print (sales.head())

              loc_property_city  loc_property_zip  loc_longitude  \
property_id                                                        
1011000170000        BARRINGTON           60010.0     -88.140521   
1011000190000        BARRINGTON           60010.0     -88.140083   
1011000250000        BARRINGTON           60010.0     -88.138667   
1011000430000        BARRINGTON           60010.0     -88.138708   
1011000460000        BARRINGTON           60010.0     -88.137997   

               loc_latitude  char_yrblt  char_bldg_sf  char_land_sf  \
property_id                                                           
1011000170000     42.153671      1893.0        1584.0        9027.0   
1011000190000     42.153577      1940.0         807.0        6699.0   
1011000250000     42.153573      1930.0        1248.0        5040.0   
1011000430000     42.153038      1935.0        1366.0        8712.0   
1011000460000     42.153036      1888.0        1823.0        7933.0   

               char_beds 

In [188]:
unmatched_index = sales.index[~sales.index.isin(property.index)]
unmatched_index2 = property.index[~property.index.isin(sales.index)]

if unmatched_index.empty and unmatched_index2.empty:
  print ("\nAll property_id can be matched between property csv and sales csv. \n")
else:
  print (unmatched_index)
  print (unmatched_index2)


All property_id can be matched between property csv and sales csv. 



In [189]:
property_list = pd.merge(sales, property, left_index=True, right_index=True, how = 'left', validate = 'many_to_one')
print (property_list.shape)
print (property_list.info())

(109307, 102)
<class 'pandas.core.frame.DataFrame'>
Index: 109307 entries, 4211120100000 to 32363060100000
Columns: 102 entries, transaction_id to meta_cdu
dtypes: bool(4), datetime64[ns](1), float64(81), int64(12), object(4)
memory usage: 83.0+ MB
None


Check again the missing value

In [190]:
value_null3 = property_list.isnull().sum().sort_values(ascending=False)
print (value_null3[value_null3!=0])

value_null_frac = property_list.isnull().mean().sort_values(ascending=False)
print ('\n',value_null_frac[value_null_frac!=0])

loc_tax_special_service_area_num                    106181
loc_school_unified_district_geoid                   105378
loc_tax_tif_district_num                            100073
loc_chicago_community_area_name                      65215
loc_chicago_ward_num                                 65215
char_tp_dsgn                                         58644
char_tp_plan                                         30998
prox_avg_school_rating_in_half_mile                  27086
acs5_median_household_total_occupied_year_built      24145
char_gar1_cnst                                       17977
other_school_district_elementary_avg_rating           4250
other_school_district_secondary_avg_rating            4057
loc_school_elementary_district_geoid                  3929
loc_school_secondary_district_geoid                   3929
acs5_median_household_renter_occupied_gross_rent      2770
char_apts                                              969
prox_num_foreclosure_per_1000_pin_past_5_years         8

Handle missing value, according to fraction and dictionary

In [191]:
# match codes with variables that have missing value
match_code = (
    var_dict
    .set_index("variable")
    .reindex(value_null_frac.index)[["type", "codes", "definition", "missing_fraction"]]
)
print (match_code[match_code["codes"].notna()]["codes"])

# char_type_resd, dictionary provides 9.9 = Missing
property_list['char_type_resd'] = property_list['char_type_resd'].fillna(9.9)


char_tp_dsgn                                         1 = Yes; 2 = No
char_tp_plan                           1 = Architect; 2 = Stock Plan
char_gar1_cnst     1 = Frame; 2 = Masonry; 3 = Frame + Masonry; 4...
char_apts          1 = Two; 2 = Three; 3 = Four; 4 = Five; 5 = Si...
char_gar1_att                                        1 = Yes; 2 = No
char_gar1_area                                       1 = Yes; 2 = No
char_type_resd     1 = 1 Story; 2 = 2 Story; 3 = 3 Story +; 4 = S...
char_porch         0 = None; 1 = Frame Enclosed; 2 = Masonry Encl...
char_attic_fnsh               1 = Living Area; 2 = Partial; 3 = None
char_heat          1 = Warm Air Furnace; 2 = Hot Water Steam; 3 =...
char_bsmt_fin      1 = Formal Rec Room; 2 = Apartment; 3 = Unfini...
char_gar1_size     1 = 1 cars; 2 = 1.5 cars; 3 = 2 cars; 4 = 2.5 ...
char_air                         1 = Central A/C; 2 = No Central A/C
char_bsmt                 1 = Full; 2 = Slab; 3 = Partial; 4 = Crawl
char_roof_cnst     1 = Shingle + A

In [197]:
print ('\n',match_code[match_code["definition"].notna()]["definition"])

# missing in loc_chicago_ward_num means outside Chicago
# meanwhile, loc_property_city provide located city, including Chicago
# Also, we check loc_chicago_community_area_name's relation with loc_chicago_ward_num, since both have same fraction
diff2 = property_list["loc_chicago_ward_num"].notna().equals(property_list["loc_chicago_community_area_name"].notna())
print ('\nWhether missing value rows are the same between loc_chicago_ward_num and loc_chicago_community_area_name: ',diff2)

# So both missing value can use a similar rule to fill
# However, for loc_property_city and loc_chicago_ward_num, one shows Chicago but the other is not
identify1 = property_list["loc_chicago_ward_num"]!=0
identify2 = property_list["loc_property_city"]!=0
diff3 = property_list[identify1 & identify2]["loc_property_city"] == 'CHICAGO'
print ('\nWhether loc_chicago_ward_num notna value rows have CHICAGO label in loc_property_city: ',diff3.all())




 loc_tax_special_service_area_num                                          Special Service Area Number
loc_school_unified_district_geoid                                       School Unified District GEOID
loc_tax_tif_district_num                                                          TIF District Number
loc_chicago_community_area_name                                           Chicago Community Area Name
loc_chicago_ward_num                                Chicago ward code. Usually missing outside Chi...
                                                                          ...                        
acs5_percent_employment_unemployed                          Percent Population Employment, Unemployed
acs5_percent_household_total_occupied_w_sel_cond    Percent Occupied Households, Total, One or Mor...
meta_class                                                                             Property Class
meta_nbhd_code                                                                  

In [199]:
# we combine loc_chicago_ward_num and loc_property_city to set a new column identifier is_Chicago
is_Chicago = (property_list["loc_chicago_ward_num"].notna()
| (property_list["loc_property_city"].str.strip().str.lower() == 'chicago'))

property_list['is_Chicago'] = is_Chicago.astype(int)

# Since loc_property_city only has 2 missing value, we replace it with CHICAGO or Unknown
city = property_list["loc_property_city"].astype("object")

mask_city_missing = city.isnull()
city[mask_city_missing & (property_list["is_Chicago"] == 1)] = "CHICAGO"
city[mask_city_missing & (property_list["is_Chicago"] == 0)] = "UNKNOWN"

property_list["loc_property_city"] = city

# Since loc_chicago_ward_num has 59% missing fraction, we have 2 choices to replace NaN
ward = property_list["loc_chicago_ward_num"].astype("object")

mask_A = ward.isnull() & (property_list["is_Chicago"] == 1)
ward[mask_A] = "Chicago-Unknown"
mask_B = ward.isnull() & (property_list["is_Chicago"] == 0)
ward[mask_B] = "Non-Chicago"

property_list["loc_chicago_ward_num"] = ward

# Similarly, loc_chicago_community_area_name can follow the same rule to replace NaN
Area = property_list["loc_chicago_ward_num"].astype("object")

mask_A = ward.isnull() & (property_list["is_Chicago"] == 1)
Area[mask_A] = "Chicago-Unknown"
mask_B = ward.isnull() & (property_list["is_Chicago"] == 0)
Area[mask_B] = "Non-Chicago"

property_list["loc_chicago_ward_num"] = Area

